# Boundary-value uncertainty

Team project by Akash Yadav, Taiwo A. Adebiyi, and Nafeezat Ajenifuja.

This output-free walkthrough preserves the numerical calculations in [the original notebook](original/UQPINNs_EX2_BC.ipynb). See [numerical notes](../docs/numerical_notes.md) for the residual, uncertainty interpretation, and recorded environment.


## 1. Runtime and reproducibility

Use TensorFlow 2.16.1 in a fresh Python 3.10 environment. The TensorFlow seed 1234 is new to this cleaned version; it cannot replay the original unseeded network initialization. The unused, absent `plotting` helper is omitted.


In [ ]:
from pathlib import Path
import os
import timeit

os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

import tensorflow.compat.v1 as tf
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
from scipy import stats

tf.disable_v2_behavior()
tf.reset_default_graph()
tf.set_random_seed(1234)
np.random.seed(1234)

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "provenance" / "manifest.json").is_file()
)
OUTPUT_DIR = ROOT / "figures" / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("TensorFlow:", tf.__version__)


## 2. Generator, encoder, discriminator, and physics residual

The original model class is retained, with device-placement logging disabled. Its second-order physical-coordinate residual is `u_xx + 5*x`; inputs are standardized internally. The encoder penalty reconstructs the latent input from the generated observation.


In [ ]:
class ODE_UQPINN:
    # Initialize the class
    def __init__(self, X_f, X_u, Y_u, layers_P, layers_Q, layers_T, lam = 1.0, beta = 1.0): 
        # X_f: collocation points, X_u: input data X, Y_u: output data Y,
        # layers_P: decoder network structure, layers_Q: encoder network structure, layers_T: discriminator network structure,
        # lam: entropic regularization parameter, beta: physics-informed regularization parameter
                
        # Normalize data
        self.Xmean = X_f.mean(0)
        self.Xstd = X_f.std(0)
        self.Ymean = Y_u.mean(0)
        self.Ystd = Y_u.std(0)
        X_f = (X_f - self.Xmean) / self.Xstd
        X_u = (X_u - self.Xmean) / self.Xstd
        # Jacobian with respect to the normalization
        self.Jacobian = 1 / self.Xstd
        
        # Normalized and collocation points and training input, output data
        self.X_f = X_f
        self.X_u = X_u
        self.Y_u = Y_u
        
        # Neural network structures
        self.layers_P = layers_P
        self.layers_Q = layers_Q
        self.layers_T = layers_T
        
        # Input, output and latent variable dimensions
        self.X_dim = X_u.shape[1]
        self.Y_dim = Y_u.shape[1]
        self.Z_dim = layers_Q[-1]
        
        # Regularization parameters
        self.lam = lam
        self.beta = beta

        # # material property
        # self.E = E
        # self.I = I
        
        # Ratio of training for generator and discriminator in each iteration: k1 for discriminator, k2 for generator
        self.k1 = 1
        self.k2 = 5

        # Initialize network weights and biases        
        self.weights_P, self.biases_P = self.initialize_NN(layers_P)
        self.weights_Q, self.biases_Q = self.initialize_NN(layers_Q)
        self.weights_T, self.biases_T = self.initialize_NN(layers_T)
        
        # Define Tensorflow session
        self.sess = tf.Session(config=tf.ConfigProto(log_device_placement=False))
        
        # Define placeholders and computational graph
        self.X_u_tf = tf.placeholder(tf.float32, shape=(None, self.X_dim))
        self.X_f_tf = tf.placeholder(tf.float32, shape=(None, self.X_dim))
        self.Y_u_tf = tf.placeholder(tf.float32, shape=(None, self.Y_dim))
        self.Z_u_tf = tf.placeholder(tf.float32, shape=(None, self.Z_dim))
        self.Z_f_tf = tf.placeholder(tf.float32, shape=(None, self.Z_dim))

        # Predict output
        self.Y_u_pred = self.net_P(self.X_u_tf, self.Z_u_tf)
        # Predict the residual at collocation points
        self.Y_f_pred = self.get_r(self.X_f_tf, self.Z_f_tf)

        # Generator loss (to be minimized)
        self.G_loss, self.KL_loss, self.recon_loss, self.PDE_loss  = self.compute_generator_loss(self.X_u_tf, self.Y_u_tf, self.Y_u_pred,
                                                                        self.X_f_tf, self.Y_f_pred, self.Z_u_tf, self.Z_f_tf)
                                                
        # Discriminator loss (to be minimized)
        self.T_loss  = self.compute_discriminator_loss(self.X_u_tf, self.Y_u_tf, self.Z_u_tf)
        
        # Generate samples of y given x by sampling from latent space
        self.sample = self.sample_generator(self.X_u_tf, self.Z_u_tf)

        # Define optimizer        
        self.optimizer_KL = tf.train.AdamOptimizer(1e-4)
        self.optimizer_T = tf.train.AdamOptimizer(1e-4)
        
        # Define train Ops
        self.train_op_KL = self.optimizer_KL.minimize(self.G_loss, 
                                                      var_list = [self.weights_P, self.biases_P,
                                                                  self.weights_Q, self.biases_Q])
                                                                    
        self.train_op_T = self.optimizer_T.minimize(self.T_loss,
                                                    var_list = [self.weights_T, self.biases_T])

        # Initialize Tensorflow variables
        init = tf.global_variables_initializer()
        self.sess.run(init)

    
    # Initialize network weights and biases using Xavier initialization
    def initialize_NN(self, layers):      
        # Xavier initialization
        def xavier_init(size):
            in_dim = size[0]
            out_dim = size[1]
            xavier_stddev = 1. / np.sqrt((in_dim + out_dim) / 2.)
            return tf.Variable(tf.random_normal([in_dim, out_dim], dtype=tf.float32) * xavier_stddev, dtype=tf.float32)   
        
        weights = []
        biases = []
        num_layers = len(layers) 
        for l in range(0,num_layers-1):
            W = xavier_init(size=[layers[l], layers[l+1]])
            b = tf.Variable(tf.zeros([1,layers[l+1]], dtype=tf.float32), dtype=tf.float32)
            weights.append(W)
            biases.append(b)        
        return weights, biases
           
           
    # Evaluates the forward pass
    def forward_pass(self, H, layers, weights, biases):
        num_layers = len(layers)
        for l in range(0,num_layers-2):
            W = weights[l]
            b = biases[l]
            H = tf.tanh(tf.add(tf.matmul(H, W), b))
        W = weights[-1]
        b = biases[-1]
        H = tf.add(tf.matmul(H, W), b)
        return H
    
    # Forcing term (right hand of the ODE)
    def f(self, X_normalized): #
        X = self.Xstd * X_normalized + self.Xmean
        return 5*X
    
    # Decoder: p(y|x,z)
    def net_P(self, X, Z):
        Y = self.forward_pass(tf.concat([X, Z], 1),
                              self.layers_P,
                              self.weights_P,
                              self.biases_P)
        return Y
    
    # Encoder: q(z|x,y)
    def net_Q(self, X, Y):
        Z = self.forward_pass(tf.concat([X, Y], 1),
                              self.layers_Q,
                              self.weights_Q,
                              self.biases_Q)
        return Z
    
    # Discriminator
    def net_T(self, X, Y):
        T = self.forward_pass(tf.concat([X, Y], 1),
                              self.layers_T,
                              self.weights_T,
                              self.biases_T)        
        return T
    
    # Physics-Informed neural network prediction
    def get_u(self, X, Z):
        z_prior = Z       
        u = self.net_P(X, z_prior)
        return u

    # Physics-Informed residual on the collocation points
    def get_r(self, X, Z):
        z_prior = Z    
        u = self.net_P(X, z_prior)
        u_x = tf.gradients(u, X)[0]
        u_xx = tf.gradients(u_x, X)[0]
        f = self.f(X)
        r = (self.Jacobian ** 2) * u_xx + f
        return r    
    
    # Compute the generator loss
    def compute_generator_loss(self, X_u, Y_u, Y_u_pred, X_f, Y_f_pred, Z_u, Z_f):  
        # Prior: z_u for input data, z_f for collocation points
        z_u_prior = Z_u
        z_f_prior = Z_f
        # Encoder: q(z|x,y)
        z_u_encoder = self.net_Q(X_u, Y_u_pred)
        z_f_encoder = self.net_Q(X_f, Y_f_pred)
        # Discriminator loss
        Y_pred = self.net_P(X_u, Z_u)
        T_pred = self.net_T(X_u, Y_pred)
        
        # KL-divergence between the data distribution and the model distribution
        KL = tf.reduce_mean(T_pred)

        # Entropic regularization
        log_q = - tf.reduce_mean(tf.square(z_u_prior-z_u_encoder))
        
        # Physics-informed loss
        loss_f = tf.reduce_mean(tf.square(Y_f_pred))

        # Generator loss
        loss = KL + (1.0-self.lam)*log_q + self.beta * loss_f
        
        return loss, KL, (1.0-self.lam)*log_q, self.beta * loss_f
    
    # Compute the discriminator loss
    def compute_discriminator_loss(self, X, Y, Z): 
        # Prior: p(z)
        z_prior = Z
        # Decoder: p(y|x,z)
        Y_pred = self.net_P(X, z_prior)                
        
        # Discriminator loss
        T_real = self.net_T(X, Y)
        T_fake = self.net_T(X, Y_pred)
        
        T_real = tf.sigmoid(T_real)
        T_fake = tf.sigmoid(T_fake)
        
        T_loss = -tf.reduce_mean(tf.log(1.0 - T_real + 1e-8) + \
                                 tf.log(T_fake + 1e-8)) 
        
        return T_loss
           
    
    # Fetches a mini-batch of data
    def fetch_minibatch(self,X, Y, N_batch):
        N = X.shape[0]
        idx = np.random.choice(N, N_batch, replace=False)
        X_batch = X[idx,:]
        Y_batch = Y[idx,:]
        return X_batch, Y_batch
    
    
    # Trains the model
    def train(self, nIter = 20000, N_u = 2, N_f = 100): 

        start_time = timeit.default_timer()
        for it in range(nIter):     

            # Sampling from the latent space for data and collocation points
            Z_u = np.random.randn(N_u, self.Z_dim)
            Z_f = np.random.randn(N_f, self.Z_dim)

            # Define a dictionary for associating placeholders with data
            tf_dict = {self.X_u_tf: self.X_u, self.Y_u_tf: self.Y_u, self.X_f_tf: self.X_f,
                     self.Z_u_tf: Z_u, self.Z_f_tf: Z_f}  
            
            # Run the Tensorflow session to minimize the loss
            for i in range(self.k1):
                self.sess.run(self.train_op_T, tf_dict)
            for j in range(self.k2):
                self.sess.run(self.train_op_KL, tf_dict)
        
            # Print
            if it % 100 == 0:
                elapsed = timeit.default_timer() - start_time
                loss_KL_value, reconv, loss_PDE = self.sess.run([self.KL_loss, self.recon_loss, self.PDE_loss], tf_dict)
                loss_T_value = self.sess.run(self.T_loss, tf_dict)
                print('It: %d, KL_loss: %.2e, Recon_loss: %.2e, PDE_loss: %.2e, T_loss: %.2e, Time: %.2f' % 
                      (it, loss_KL_value, reconv, loss_PDE, loss_T_value, elapsed))
                start_time = timeit.default_timer()
                
    # Generate samples of y given x by sampling from the latent space z
    def sample_generator(self, X, Z):        
        # Prior:
        z_prior = Z       
        # Decoder: p(y|x,z)
        Y_pred = self.net_P(X, z_prior)      
        return Y_pred

    # Predict y given x
    def generate_sample(self, X_star):
        # Normalize the input data
        X_star = (X_star - self.Xmean) / self.Xstd
        Z = np.random.randn(X_star.shape[0], self.Z_dim)
        tf_dict = {self.X_u_tf: X_star, self.Z_u_tf: Z}       
        Y_star = self.sess.run(self.sample, tf_dict) 
        Y_star = Y_star 
        return Y_star

## 3. Synthetic observations and reference

Twenty paired endpoint observations use shared Gaussian offsets within each pair. The data cell retains the original NumPy seed 42 and 200 collocation points.


In [ ]:
N_f = 200
N_ref = 200
N_u = 20
X_dim = 1
Y_dim = 1
Z_dim = 1
E = 20000000.0
I = 1e-06
mu_eta = 0.0
sigma_eta = 5e-06

def f(X, E, I):
    return -(50 / (3 * E * I)) * X ** 3
X_f = np.linspace(0.0, 1.0, N_f)[:, None]
X_ut = np.linspace(0.0, 1.0, 2)[:, None]
X_u = X_ut
for i in range(N_u - 1):
    X_u = np.vstack((X_u, X_ut))
print(np.size(X_u))
np.random.seed(42)
eta_samples = np.random.normal(mu_eta, sigma_eta, 1)
Y_ut = f(X_ut, E, I) - eta_samples
Y_u = Y_ut
for i in range(N_u - 1):
    eta_samples = np.random.normal(mu_eta, sigma_eta, 1)
    Y_ut = f(X_ut, E, I) - eta_samples
    Y_u = np.vstack((Y_u, Y_ut))
print(np.size(Y_u))
X_ref = np.linspace(0.0, 1.0, N_ref)[:, None]
Y_ref = f(X_ref, E, I)


## 4. Network configuration

Generator and encoder: four hidden layers of width 50. Discriminator: two hidden layers of width 50. Hidden activations are tanh, with linear outputs; the discriminator loss applies a sigmoid.


In [ ]:
# Model creation
layers_P = np.array([X_dim+Z_dim,50,50,50,50,Y_dim])
layers_Q = np.array([X_dim+Y_dim,50,50,50,50,Z_dim])
layers_T = np.array([X_dim+Y_dim,50,50,1])

# Define the model
model = ODE_UQPINN(X_f, X_u, Y_u, layers_P, layers_Q, layers_T, lam = 1.5, beta = 1.)

## 5. Alternating full-batch training

The original setting is 30,000 iterations, each with one discriminator and five generator/encoder updates. Both Adam learning rates are 1e-4. A short CLI validation run is not a completed experiment.


In [ ]:
# Train the model
model.train(nIter = 30000, N_u = 2*N_u, N_f = N_f)

## 6. Pointwise predictive sampling

Use 2,000 sampling passes on 200 reference points to estimate the predictive mean and variance. Each pass samples an independent latent value for each coordinate; these are not coherent function-path draws.


In [ ]:
# Prediction

# Make prediciton at randomly chosen points X_ff (black lines are generated samples, red stars are predicted mean)
N_samples = 2000
samples_mean = np.zeros((X_ref.shape[0], N_samples))
for i in range(0, N_samples):
    samples_mean[:,i:i+1] = model.generate_sample(X_ref)

# Compute the mean and the variance of the prediction at randomly selected points in the domain
mu_pred = np.mean(samples_mean, axis = 1)    
Sigma_pred = np.var(samples_mean, axis = 1)


## 7. Visualization

The shaded band is the sampled mean plus or minus two standard deviations. Coverage is not calibrated here. The original vertical-axis scaling label is retained; verify its physical units before quantitative reuse. New figures go to the ignored `figures/generated/` directory, leaving the historical figure unchanged.


In [ ]:
# Plot the prediction with the uncertainty versus the reference solution

ax = plt.figure(2,figsize=(7,5))
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.plot(X_u, Y_u, 'kx', markersize = 4, label = "Data points")
lower = mu_pred - 2.0*np.sqrt(Sigma_pred)
upper = mu_pred + 2.0*np.sqrt(Sigma_pred)
plt.fill_between(X_ref.flatten(), lower.flatten(), upper.flatten(), 
                  facecolor='orange', alpha=0.5, label="Two std band")
plt.plot(X_ref,Y_ref,'b-', label = "Exact", linewidth=2)
plt.plot(X_ref, mu_pred, 'r--', label = "Prediction", linewidth=2)
plt.xlabel('X (m)',fontsize=14)
plt.ylabel('Deflection (x $10^{-4}$ m)',fontsize=14)
plt.legend(loc='lower left', frameon=True, prop={'size': 13})
# Save the figure as a PNG file
plt.savefig(OUTPUT_DIR / "boundary_uncertainty.png", dpi=300)
plt.show()

## 8. Predictive-mean comparison

The original relative Euclidean error compares the predictive mean with the nominal cubic reference. It does not evaluate distributional accuracy or uncertainty coverage. Execute from a fresh runtime.


In [ ]:
# Compute the prediction relative error 

mu_pred = mu_pred[:,None]
error_u = np.linalg.norm(Y_ref-mu_pred,2)/np.linalg.norm(Y_ref,2)
print('Error u: %e' % (error_u))  